# ML10 · Autoencoder 自编码器

用神经网络把高维数据压成精简的潜在表示（latent representation），再还原回来，藉「重建得有多像」学会数据的重要结构。

## 学习目标
- 理解自编码器（autoencoder）的整体结构：编码器（encoder）与解码器（decoder）的串接。
- 认识瓶颈（bottleneck）与潜在表示（latent representation）的角色，知道压缩为何能逼出重点。
- 理解重建损失（reconstruction loss）如何驱动训练，让输出逼近输入。
- 能用自造的多维特征数据，创建并训练一个最小自编码器。
- 会可视化重建误差（reconstruction error），判读哪些样本被还原得好或不好。
- 理解潜在维度（latent dimension）大小对压缩率与重建品质的取舍。

In [ ]:
# 概念：环境与字体 setup（之后会画损失曲线、长条图与散布图）
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

# 让中文标题与标签能正常显示（依系统挑一个常见字体；找不到就用预设）
matplotlib.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "PingFang TC", "Noto Sans CJK TC", "SimHei", "Arial Unicode MS"]
matplotlib.rcParams["axes.unicode_minus"] = False   # 负号用一般减号显示，避免变成方框

print("numpy 版本:", np.__version__)

## 为什么要压缩：从监督到自监督

前面的分类模型需要人工标签当答案；自编码器改用自监督学习（self-supervised learning），让「数据自己当答案」，目标是把输入原样还原回来。

为什么有用：真实数据常有冗余，字段彼此相关（例如楼高、楼层数、总面积会一起变动）。压缩（降维 dimensionality reduction）迫使模型只保留重要信息，这正是表示学习（representation learning）的起点。

何时用：没有标签、想找数据内在结构、想用少量数字概括一笔数据时。

In [ ]:
# 概念：自造一批带冗余字段的多维特征，观察字段高度相关（彼此可互相推算）就是「可被压缩」的信号
import numpy as np

rng = np.random.default_rng(0)   # 固定乱数种子，让每次结果一致方便对照

# 造 200 栋楼房，先生出一个潜在的「规模因子」，多数字段都由它衍生（刻意制造冗余）
n = 200
scale = rng.uniform(1.0, 5.0, size=n)            # 规模因子：越大代表楼越大
floors = np.round(scale * 4 + rng.normal(0, 1, n))      # 楼层数约与规模成正比
height = floors * 3.0 + rng.normal(0, 1.5, n)           # 楼高约等于楼层数乘层高
floor_area = scale * 80 + rng.normal(0, 10, n)          # 单层面积也跟着规模走
total_area = floor_area * floors                        # 总楼地板几乎可由前两栏算出

features = np.column_stack([floors, height, floor_area, total_area])
print("特征 shape (样本数, 特征数):", features.shape)

# 相关系数越接近 1 代表字段越冗余；高相关就是压缩空间的来源
corr = np.corrcoef(features, rowvar=False)   # rowvar=False：每栏当一个变量
print("字段间相关系数矩阵（越接近 1 越冗余）:")
print(np.round(corr, 2))

## 自编码器结构：编码器与解码器

自编码器是两段网络的串接：编码器（encoder）把输入一路往下压到很窄的中间层，解码器（decoder）再把它展开回原本的尺寸。整体像「漏斗再展开」。

关键：输入与输出的维度必须相同，因为训练目标是「输出要等于输入」。前向传递（forward pass）就是数据从输入依序穿过每一层、算到输出的过程。

常见是对称结构（symmetric architecture），例如 8 → 4 → 2 → 4 → 8，左右维度镜像对齐。

In [ ]:
# 概念：用 list 描述各层维度，仿真一笔数据做前向传递，观察「形状先变窄再变回来」
import numpy as np

# 各层输出维度：左半是编码器（越来越窄），中间是瓶颈，右半是解码器（镜像展开）
layer_dims = [8, 4, 2, 4, 8]

rng = np.random.default_rng(1)
x = rng.normal(size=8)            # 造一笔 8 维输入当示范

# 为每一层相邻维度造一个随机权重矩阵，仿真前向传递时形状如何被改变
activation = x
print("输入形状:", activation.shape)
for in_dim, out_dim in zip(layer_dims[:-1], layer_dims[1:]):
    W = rng.normal(size=(in_dim, out_dim)) * 0.3   # 权重把 in_dim 维映射到 out_dim 维
    activation = np.tanh(activation @ W)           # @ 是矩阵乘法；tanh 是常见的非线性活化函数
    print("经过一层 ->", in_dim, "转", out_dim, " 目前形状:", activation.shape)

# 注意：输入是 8 维、最后输出也是 8 维，输入输出同尺寸才有「重建」可言
print("最后输出形状:", activation.shape, "（与输入相同）")

## 瓶颈与潜在表示

整个网络最窄的那一层就是瓶颈（bottleneck）；通过瓶颈后得到的矢量就是潜在表示（latent representation），它是整笔数据的精简座标。

为什么瓶颈逼出重点：信息必须挤过这个窄口，模型只能保留最能帮助还原的特征，丢掉冗余，这就是信息压缩（information compression）。潜在维度（latent dimension）就是瓶颈的宽度。

直觉：瓶颈越窄压缩越狠，但压太狠会还原失真。下面用最简单的线性压缩（取数据主要方向）来感受这个取舍。

In [ ]:
# 概念：把瓶颈维度设为 1、2、3，用线性投影压缩再还原，比较还原误差，感受「压太狠会失真」
import numpy as np

rng = np.random.default_rng(2)
# 造 150 笔 4 维数据，但其实只由 2 个独立方向构成（内在维度约为 2）
base = rng.normal(size=(150, 2))
mix = rng.normal(size=(2, 4))            # 把 2 维混成 4 维（制造冗余）
data = base @ mix
data = (data - data.mean(axis=0)) / data.std(axis=0)   # 先标准化，让各栏尺度一致

# 技巧：用 SVD 取数据的主要方向当「线性瓶颈」，是自编码器压缩的线性版直觉
U, S, Vt = np.linalg.svd(data, full_matrices=False)

for latent_dim in [1, 2, 3]:
    directions = Vt[:latent_dim]                 # 取前 latent_dim 个主要方向
    latent = data @ directions.T                 # encode：投影到瓶颈维度
    reconstructed = latent @ directions          # decode：用同方向展开回 4 维
    mse = np.mean((data - reconstructed) ** 2)   # 重建误差（均方误差）
    print(f"瓶颈维度 = {latent_dim}  ->  平均重建误差 = {mse:.4f}")

# 注意：因为真实内在维度约为 2，瓶颈到 2 之后误差就大幅下降，再加维度收益有限

## 重建损失与训练流程

训练目标只有一句话：让输出逼近输入。衡量「逼近程度」的就是重建损失（reconstruction loss），最常用均方误差（mean squared error, MSE）：把每个元素的误差平方后取平均。

怎么变好：用梯度下降（gradient descent），每个 epoch 算出损失对权重的梯度，往让损失下降的方向微调权重，一步步把损失降下来。一个训练循环（training loop）就是反复做「前向算损失 → 反向算梯度 → 更新权重」。

公式：MSE = 平均((输入 − 输出)^2)。

In [ ]:
# 概念：手刻一个最小线性自编码器，用 MSE 当损失、用梯度下降训练数十个 epoch，收集并画出损失曲线
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(3)
# 造 300 笔 6 维数据，内在维度约 2（方便压到很窄仍能还原）
base = rng.normal(size=(300, 2))
X = base @ rng.normal(size=(2, 6))
X = (X - X.mean(axis=0)) / X.std(axis=0)   # 标准化是稳定训练的常见前处理

input_dim, latent_dim = 6, 2
# 编码器与解码器各用一个权重矩阵（线性自编码器，无偏置与非线性，聚焦训练直觉）
W_enc = rng.normal(size=(input_dim, latent_dim)) * 0.1
W_dec = rng.normal(size=(latent_dim, input_dim)) * 0.1

lr = 0.05            # 学习率：每步往下走的幅度
losses = []          # 收集每个 epoch 的损失，之后画曲线
for epoch in range(80):
    latent = X @ W_enc            # 前向：编码
    recon = latent @ W_dec        # 前向：解码
    error = recon - X             # 重建误差（输出减输入）
    loss = np.mean(error ** 2)    # MSE：误差平方的平均
    losses.append(loss)

    # 反向：用链锁法则算损失对两个权重的梯度（除以样本数做平均）
    grad_dec = latent.T @ error / X.shape[0]
    grad_enc = X.T @ (error @ W_dec.T) / X.shape[0]
    W_dec -= lr * grad_dec        # 往梯度反方向更新，让损失下降
    W_enc -= lr * grad_enc

print(f"第一个 epoch 损失: {losses[0]:.4f}   最后一个 epoch 损失: {losses[-1]:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(losses)
plt.xlabel("epoch")
plt.ylabel("重建损失 MSE")
plt.title("训练过程中重建损失逐步下降")
plt.show()

## 用自造特征示范压缩与还原

训练好的自编码器可以拆成两段分开用：先用编码器做 encode 得到潜在矢量，需要时再用解码器 decode 还原。实用价值是「平常只存小矢量，要用时再展开」。

压缩率（compression ratio）= 原始维度 / 潜在维度，数字越大代表存得越省。先做标准化（normalization）让各栏尺度一致，训练才稳定。

下面接续上一节训练好的权重，取几笔样本并排比较原始与还原数值。

In [ ]:
# 概念：把模型拆成 encode / decode 两段使用，并计算压缩率与还原相似度
import numpy as np

rng = np.random.default_rng(3)
# 重现上一节同一份数据与训练（自足，不依赖上一个 cell 的变量）
base = rng.normal(size=(300, 2))
X = base @ rng.normal(size=(2, 6))
X = (X - X.mean(axis=0)) / X.std(axis=0)

input_dim, latent_dim = 6, 2
W_enc = rng.normal(size=(input_dim, latent_dim)) * 0.1
W_dec = rng.normal(size=(latent_dim, input_dim)) * 0.1
lr = 0.05
for epoch in range(80):
    latent = X @ W_enc
    recon = latent @ W_dec
    error = recon - X
    grad_dec = latent.T @ error / X.shape[0]
    grad_enc = X.T @ (error @ W_dec.T) / X.shape[0]
    W_dec -= lr * grad_dec
    W_enc -= lr * grad_enc

def encode(batch):
    return batch @ W_enc          # 只走编码器：高维 -> 潜在矢量

def decode(codes):
    return codes @ W_dec          # 只走解码器：潜在矢量 -> 还原高维

sample = X[:3]                    # 取前 3 笔当示范
codes = encode(sample)            # 存下来的就是这个小矢量
restored = decode(codes)          # 需要时再展开

compression_ratio = input_dim / latent_dim
print("压缩率 (原始维度 / 潜在维度):", compression_ratio)
for i in range(3):
    print(f"样本 {i}  潜在矢量:", np.round(codes[i], 3))
    print("   原始:", np.round(sample[i], 2))
    print("   还原:", np.round(restored[i], 2))

## 视觉化重建误差与潜在空间

对每一笔资料算出它的重建误差（reconstruction error，自己这笔的 MSE），画成长条图就能看出误差分布（error distribution）。误差特别大的样本往往是「不寻常」的，这就是异常侦测的直觉（anomaly intuition）。

把每笔的 2 维潜在向量画成散布图（latent scatter），相似的样本通常聚在一起、不同类别会分开。

下面对全部样本算逐笔误差画长条图，再把潜在向量画成依类别上色的散布图。

In [ ]:
# 概念：算每笔重建误差画长条图，并把 2 维潜在矢量画成依类别上色的散布图
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(4)
# 造两群楼房：群 0 规模较小、群 1 规模较大，各 80 笔（潜在类别）
group0 = rng.normal(loc=[-1.5, -1.0], scale=0.4, size=(80, 2))
group1 = rng.normal(loc=[1.5, 1.0], scale=0.4, size=(80, 2))
base = np.vstack([group0, group1])
labels = np.array([0] * 80 + [1] * 80)        # 类别只用来上色，训练时不使用（自监督）
X = base @ rng.normal(size=(2, 6))
X = (X - X.mean(axis=0)) / X.std(axis=0)

# 训练同款线性自编码器
input_dim, latent_dim = 6, 2
W_enc = rng.normal(size=(input_dim, latent_dim)) * 0.1
W_dec = rng.normal(size=(latent_dim, input_dim)) * 0.1
for epoch in range(120):
    latent = X @ W_enc
    recon = latent @ W_dec
    error = recon - X
    W_dec -= 0.05 * (latent.T @ error / X.shape[0])
    W_enc -= 0.05 * (X.T @ (error @ W_dec.T) / X.shape[0])

latent = X @ W_enc
recon = latent @ W_dec
per_sample_error = np.mean((recon - X) ** 2, axis=1)   # axis=1：每一笔自己的平均误差

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(range(len(per_sample_error)), per_sample_error)
axes[0].set_xlabel("样本索引")
axes[0].set_ylabel("重建误差")
axes[0].set_title("每笔样本的重建误差分布")

scatter = axes[1].scatter(latent[:, 0], latent[:, 1], c=labels, cmap="coolwarm", s=20)
axes[1].set_xlabel("潜在维度 1")
axes[1].set_ylabel("潜在维度 2")
axes[1].set_title("潜在空间散布图（依类别上色）")
plt.tight_layout()
plt.show()

print("平均重建误差:", round(per_sample_error.mean(), 4))

## 练习

以下三题由浅入深，皆为复合型但简短。数据请自己用 numpy 造（仿真即可），完成后对照「验收」确认输出。

In [ ]:
# TODO 1 ·（简单）楼房特征的最小压缩（集成：编码器/解码器 + 重建损失）
#   情境：自造一批楼房数据，字段含楼高、楼层数、单层面积等彼此相关的多维特征。
#   要求：
#     1. 用 numpy 自造数据并逐栏标准化，搭一个对称的小型自编码器（如 6 -> 2 -> 6）。
#     2. 以均方误差为重建损失训练数十个 epoch，记录并画出损失曲线。
#   验收：应该看到损失随 epoch 明显下降，且部分样本的还原值接近原始值。
# TODO: 在这里完成


In [ ]:
# TODO 2 ·（中间）潜在维度的取舍实验（集成：瓶颈 + 潜在表示 + 重建误差 + 压缩率）
#   情境：用自造的社区地块数据（面积、容积率、绿地比例、退缩距离等多字段）做压缩分析。
#   要求：
#     1. 将瓶颈维度分别设为 1、2、3 各训练一次。
#     2. 对每种设置计算平均重建误差与压缩率（原始维度 / 潜在维度）。
#     3. 把「潜在维度 对 重建误差」画成折线图。
#   验收：应看到潜在维度越大、重建误差越小，并能说出一个「够用又省」的维度选择。
# TODO: 在这里完成


In [ ]:
# TODO 3 ·（稍难）用重建误差找异常街廓（集成：encode/decode + 重建误差 + 潜在散布 + 异常直觉）
#   情境：自造一批「正常」街廓特征训练模型，另外混入少量刻意偏离的异常街廓（如异常高的容积率）。
#   要求：
#     1. 只用正常数据训练自编码器，再对全部数据（含异常）做 encode/decode。
#     2. 计算每笔重建误差，设一个门槛把高误差样本标为可疑（例如取正常误差的某个百分位）。
#     3. 把 2 维潜在矢量画成散布图，标出被判为异常的点。
#   验收：应看到刻意混入的异常街廓多半落在高重建误差区、并在潜在散布图中偏离主群；
#         并能讨论门槛怎么定较合理。
# TODO: 在这里完成


## 小结

- 自编码器是自监督学习：不用标签，让数据自己当答案，目标是把输入原样还原回来。
- 结构是编码器与解码器的串接（常为对称），输入与输出维度相同才有「重建」可言。
- 最窄的瓶颈层产生潜在表示；压缩迫使模型丢掉冗余、留下本质，潜在维度就是瓶颈宽度。
- 训练用重建损失（多用 MSE）驱动，靠梯度下降在训练循环里一步步把损失降下来。
- 模型可拆成 encode / decode 两段使用：平常存小矢量、需要时再展开，压缩率为原始维度除以潜在维度。
- 逐笔重建误差能找出「不寻常」的样本（异常直觉）；把潜在矢量画成散布图可看出相似样本是否聚在一起。
- 潜在维度的选择是取舍：太窄会失真、太宽不够省，挑「够用又省」的维度最实用。